# Session 9 — Regression Basics

**Goal:** learn the modelling tool itself — fit, interpret, evaluate, and diagnose a
linear regression — on a deliberately low-stakes target, so that Session 10 can build
the actual disease classifier without also learning regression from scratch.

## What this stage does for the system

Sessions 5-8 screened inputs one at a time. Every one of those methods answers "does
this input relate to the outcome, on its own?" A model answers a different question:
**given several inputs at once, what is the best prediction, and what does each input
contribute once the others are accounted for?**

This session predicts `thalach` (max heart rate) from `age` and other inputs rather
than predicting disease. That is deliberate. `thalach` is continuous, so ordinary least
squares applies directly; there is a well-known clinical formula to benchmark against;
and getting it wrong harms nobody, which is the right setting for learning what
residual diagnostics look like when they fail. Session 10 switches the target to
`target` and the model to logistic regression, keeping everything else.

The habit installed here is **always beat a baseline**. A model that cannot outperform
"predict the average for everyone" is not a model, and R² alone will not tell you —
Step 2 shows the standard clinical formula scoring *worse* than the average.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Two baselines to beat

Before fitting anything: what does a trivially simple predictor achieve? Two of them
here — predict the group mean for every patient, and the textbook clinical rule
`max heart rate ≈ 220 − age`. Any model that cannot beat both has earned nothing.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y = df["thalach"].to_numpy()

mean_prediction = np.full_like(y, y.mean(), dtype=float)
formula_prediction = 220 - df["age"].to_numpy()

def report(name, predictions):
    print(f"{name:22} MAE={mean_absolute_error(y, predictions):6.2f}  "
          f"RMSE={mean_squared_error(y, predictions) ** 0.5:6.2f}  "
          f"R2={r2_score(y, predictions):+.3f}")

print(f"actual thalach: mean={y.mean():.2f}, sd={y.std(ddof=1):.2f}\n")
report("predict the mean", mean_prediction)
report("220 - age formula", formula_prediction)

**Observe:** predicting the mean gives `MAE=18.50` and `R2=+0.000` by construction; the
`220 − age` formula gives `MAE=19.47` and **`R2=−0.324`** — a *negative* R².
**Infer:** a negative R² means the formula is worse than predicting the average for
every patient, which is a genuinely useful thing to have discovered before trusting it.
The reason is visible in the arithmetic: mean age here is 54.5, so `220 − age` predicts
about 165 bpm, while the registry's actual mean is `149.60` — the formula is
systematically ~16 bpm too high for this population. It was derived on healthy adults,
not on patients referred for angiography, which is Session 4's sampling lesson arriving
from the opposite direction: a rule fitted on one population misapplied to another.
Note also that R² *can* go negative, despite its name — that only surprises people who
learned it as "the square of a correlation", which it is only for a fitted least-squares
model on its own training data.

## Step 3 — Fit a simple linear regression

Ordinary least squares finds the intercept and slope minimising the sum of squared
residuals. One input to start, so the fit can be plotted and read directly.

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

X_age = df[["age"]].to_numpy()
model = LinearRegression().fit(X_age, y)

print(f"fitted:   thalach = {model.intercept_:.2f} + ({model.coef_[0]:.3f}) * age")
print(f"textbook: thalach = 220.00 + (-1.000) * age\n")
report("fitted regression", model.predict(X_age))
report("220 - age formula", formula_prediction)
report("predict the mean", mean_prediction)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["age"], y, alpha=0.45, s=22, color="steelblue", edgecolor="none", label="patients")
ages = np.linspace(df["age"].min(), df["age"].max(), 50)
ax.plot(ages, model.predict(ages.reshape(-1, 1)), "r-", lw=2, label="fitted OLS")
ax.plot(ages, 220 - ages, "k--", lw=2, label="220 - age")
ax.axhline(y.mean(), color="gray", ls=":", lw=2, label="predict the mean")
ax.set_xlabel("age (years)"); ax.set_ylabel("thalach (bpm)")
ax.set_title("Three predictors of maximum heart rate")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** the fit is `thalach = 204.15 + (−1.000) × age` with `MAE=16.99`,
`R2=+0.156` — beating both baselines — and the plot shows the fitted line running
parallel to but roughly 16 bpm below the dashed `220 − age` line.
**Infer:** the slope of `−1.000` matching the textbook's is a striking coincidence and
worth reading carefully: the *rate* of decline with age is exactly what the clinical
rule says, but the *level* is not, and least squares corrected the level while leaving
the slope alone. That is the whole story of Step 2's negative R² — the formula's
structure was right, its calibration was wrong for this population. Read the
coefficients literally: each additional year of age is associated with 1.0 fewer bpm of
maximum heart rate; the intercept of `204.15` is the fitted value at age 0, which is
meaningless as a prediction and should be treated as a fitting constant, not a
biological claim. And `R2=0.156` means age explains 16% of the variation in max heart
rate — a real effect, and nowhere near a usable predictor on its own.

## Step 4 — Multiple regression: does more complexity earn its keep?

Adding inputs can only *increase* R² on the data you fit — even inputs that are pure
noise. **Adjusted R²** penalises each added parameter, so it can go down, which makes
it the honest comparison between models of different sizes.

In [ ]:
import statsmodels.api as sm

candidate_sets = {
    "age only": ["age"],
    "+ oldpeak": ["age", "oldpeak"],
    "+ exang": ["age", "oldpeak", "exang"],
    "+ trestbps, sex": ["age", "oldpeak", "exang", "trestbps", "sex"],
}

rows = []
for name, cols in candidate_sets.items():
    fit = sm.OLS(y, sm.add_constant(df[cols])).fit()
    rows.append({"inputs": name, "k": len(cols), "R2": fit.rsquared,
                 "adj R2": fit.rsquared_adj, "AIC": fit.aic})

# A pure-noise column, to show what R2 does with an input that cannot help.
rng = np.random.default_rng(0)
noisy = df[["age", "oldpeak", "exang", "trestbps", "sex"]].assign(
    noise=rng.normal(size=len(df)))
fit_noise = sm.OLS(y, sm.add_constant(noisy)).fit()
rows.append({"inputs": "+ pure noise", "k": 6, "R2": fit_noise.rsquared,
             "adj R2": fit_noise.rsquared_adj, "AIC": fit_noise.aic})

print(pd.DataFrame(rows).round({"R2": 4, "adj R2": 4, "AIC": 1}).to_string(index=False))

**Observe:** R² climbs monotonically down the table, `0.1557 → 0.3238`, and adding a
column of *pure random noise* leaves R² unchanged to four decimals — it never decreases —
while adjusted R² falls (`0.3121 → 0.3098`) and AIC rises (`2598.6 → 2600.6`).
**Infer:** the noise row is the demonstration. R² measures fit to the data in front of
you and is mathematically incapable of decreasing when an input is added, so an R² that
went up is not evidence the input helped — here a column of pure noise held it exactly
level, and on a smaller registry it would have visibly risen. Adjusted R² and AIC both charge
for parameters and both correctly reject the noise column, which is why they are the
metrics to compare models on. Even so, they are corrections applied to training-set fit,
not measurements of performance on new patients — a model can beat every rival on
adjusted R² and still fail on unseen data. That gap is Session 10's entire subject, and
Session 11 shows how wide it gets.

## Step 5 — Coefficient significance and intervals

`scikit-learn` returns coefficients and nothing else. `statsmodels` returns the full
inferential apparatus — standard errors, t-statistics, p-values, confidence intervals —
which is Sessions 6-7's machinery applied to each coefficient in turn.

In [ ]:
inputs = ["age", "oldpeak", "exang", "trestbps", "sex"]
ols = sm.OLS(y, sm.add_constant(df[inputs])).fit()
print(ols.summary().tables[1])
print(f"\nR2 = {ols.rsquared:.3f},  adjusted R2 = {ols.rsquared_adj:.3f}")
print(f"F-test p-value (is the model better than an intercept alone?): {ols.f_pvalue:.2e}")

**Observe:** `age` (`−0.912`), `exang` (`−14.308`), and `oldpeak` (`−4.160`) all have
p-values at or below `0.000`; `trestbps` is marginal at `p = 0.026`; and **`sex` is not
significant at all** (`coef = −1.056`, `p = 0.662`, interval `[−5.81, +3.69]` spanning
zero).
**Infer:** the `sex` result is the instructive one. Session 6 found sex strongly related
to *disease*; here it adds nothing to predicting *max heart rate* once age and the
others are in the model — a reminder that a coefficient in a multivariate fit means
"the effect of this input **holding the others fixed**", which is a different quantity
from the pairwise relationships Sessions 5-8 measured, and the two can disagree without
either being wrong. The `exang` coefficient is the one to report: patients with
exercise-induced angina reach a max heart rate 14.3 bpm lower on average, all else
equal, and its interval `[−19.19, −9.43]` is comfortably clear of zero. These
coefficients are readable at face value only because Session 5 checked
multicollinearity first (all VIF below 1.4); with redundant inputs they would be
unstable and the p-values misleading.

## Step 6 — Residual diagnostics: is a straight line even the right shape?

R² says how much variation the model captured. The **residuals** — what it failed to
capture — say whether the model's *form* is right. Least squares assumes residuals with
zero mean, constant variance across the range of predictions, and no remaining pattern.

In [ ]:
predictions = ols.fittedvalues
residuals = y - predictions

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))

axes[0].scatter(predictions, residuals, alpha=0.45, s=22, color="steelblue", edgecolor="none")
axes[0].axhline(0, color="red", ls="--")
axes[0].set_xlabel("fitted value"); axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs fitted (want: shapeless band)")

from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot of residuals")

axes[2].hist(residuals, bins=30, color="steelblue", edgecolor="white")
axes[2].set_xlabel("residual"); axes[2].set_ylabel("patients")
axes[2].set_title(f"Residuals, skew = {stats.skew(residuals):+.2f}")
plt.tight_layout()
plt.show()

from statsmodels.stats.diagnostic import het_breuschpagan
bp_p = het_breuschpagan(residuals, sm.add_constant(df[inputs]))[1]
print(f"residual mean: {residuals.mean():.2e}  (zero by construction)")
print(f"Breusch-Pagan test for non-constant variance: p={bp_p:.4f}"
      f"  -> {'variance is not constant' if bp_p < 0.05 else 'no evidence against constant variance'}")
print(f"Shapiro-Wilk on residuals: p={stats.shapiro(residuals).pvalue:.2e}, "
      f"skew={stats.skew(residuals):+.2f}")

**Observe:** the residuals-vs-fitted panel is a reasonably shapeless band with no funnel
or curve; Breusch-Pagan finds no evidence against constant variance (`p = 0.532`); but
the residuals are left-skewed (`−0.73`) and Shapiro rejects Normality outright, with the
Q-Q plot bending away at the low end.
**Infer:** these three checks fail differently and matter differently, which is the
point of running all three. Constant variance passing is the important one — it is what
makes the standard errors and p-values in Step 5 valid, and a funnel shape would have
invalidated every interval there. Non-Normal residuals are the mild failure: at n=297
the CLT covers the coefficient estimates (Session 4 again), so the p-values survive; what
does *not* survive is any prediction interval for an individual patient, which assumes
Normal residuals directly. The left skew has a readable cause — a handful of patients
with unusually low max heart rate that no linear combination of these inputs explains,
the same tail Session 2's z-scores flagged. Worth checking those patients individually
before concluding the model form is wrong.

## Step 7 — The evaluation problem this session cannot solve

Every number above was computed on the same 297 patients the model was fitted to. The
model chose its coefficients to minimise error on exactly those rows, so every metric
reported so far is optimistic by an unknown amount.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

flexible = DecisionTreeRegressor(max_depth=None, random_state=0).fit(df[inputs], y)

print("Fit quality measured on the SAME data used for fitting:")
report("linear regression", ols.fittedvalues)
report("unrestricted tree", flexible.predict(df[inputs]))
print()
print("The tree is 'better' on every metric. Would you deploy it?")

**Observe:** the unrestricted decision tree achieves `MAE=0.04` and `R2=+1.000` —
effectively perfect — against the regression's `MAE=14.60` and `R2=0.324`.
**Infer:** the tree has memorised the training set, storing a leaf per patient rather
than learning anything generalisable, and it would predict poorly for patient 298 while
scoring perfectly on every number in this notebook. This is the entire argument for
Session 10: **fit quality on training data is not evidence of predictive quality**, and
no diagnostic run on the training set can distinguish a model that learned from one that
memorised. The only fix is data the model has not seen. Note the linear regression is
not immune, merely less exposed — it has six parameters where the tree effectively has
297, so it has far less capacity to memorise, but its `R2=0.324` is still an
overestimate of what it would achieve on new patients.

## What this session hands to the next one

- **A fitted, interpreted, diagnosed model**, and the workflow that produced it:
  baseline first, fit, read the coefficients with their intervals, check the residuals.
- **The multivariate reading of a coefficient** — "holding the others fixed" — which
  differs from Sessions 5-8's pairwise findings, as `sex` demonstrated.
- **Adjusted R² and AIC** as the honest way to compare models of different sizes, and
  the noise-column demonstration of why raw R² is not.
- **An unsolved problem**: every metric here is measured on training data, and Step 7
  showed how far that can be from the truth.

Session 10 switches the target to disease, the model to logistic regression, and — the
substantive change — evaluates on patients the model has never seen.

## Try it yourself

1. Add `chol` to the Step 5 model. Its pairwise relationship with `thalach` was
   effectively zero (Session 5) — does it earn a place multivariately? What happens to
   adjusted R² and AIC?
2. Replace `age` with `age**2` and with `log(age)`. Does either improve the residual
   plot's shape, and does adjusted R² agree that it helped?
3. Identify the patients with the five most negative residuals in Step 6. What do they
   have in common, and does adding an input that captures it fix the skew?
4. Refit Step 5 on a random half of the registry and compare the coefficients against
   the other half's. How stable are they — and which input's coefficient moves most?